## **OSEMN | SCRUB: Data Preprocessing + Feature Engineering**
Data preprocessing phase to the integrated IPEDS and CDS datasets. The preprocessing process includes duplicate validation, column standardisation, removal of non-analytical variables, missing value treatment and feature engineering for admissions and enrollment related key metrics for EDA and modelling.


**1. Environment Setup: Mount Drive to Upload in Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Dataset Loading and Structure Review**

Understand the data structure after data integration between IPEDS and CDS

In [ ]:
import pandas as pd

#load dataset
df = pd.read_excel('/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/finalvalidated_dataset_2019_2023.xlsx')

#check number of rows and columns
print("Shape of dataset:", df.shape)

Shape of dataset: (300, 52)


**3. Duplicate Institution-Year Check**

The integrated dataset examined for duplicate records at the institution-year level. Each institution is expected to have only one observation per year; therefore, multiple records for the same institution and year indicate data duplication issues.

**3.1 Identify duplicates**

In [ ]:
#check duplicates based on institution + year
duplicates = df[df.duplicated(subset=["unitid", "year"], keep=False)]

print("Number of duplicate rows:", duplicates.shape[0])
duplicates.head()

Number of duplicate rows: 0


,unitid,institution_name,year,level_of_student,enrollment_men,enrollment_women,idx_ef,enrollment_asian,enrollment_black,enrollment_hispanic,...,graduation_women_Bachelor's or equiv subcohort (4-yr institution) Completers of bachelor's or equiv degrees total (150% of normal time),graduation_women_Bachelor's or equiv subcohort (4-yr institution) Transfer-out students,university,filename,pdf_path,retention_rate_y,applicants,admitted,enrolled,status


**3.2 Data Types**

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 52 columns):
 #   Column                                                                                                                                   Non-Null Count  Dtype  
---  ------                                                                                                                                   --------------  -----  
 0   unitid                                                                                                                                   300 non-null    int64  
 1   institution_name                                                                                                                         300 non-null    object 
 2   year                                                                                                                                     300 non-null    int64  
 3   level_of_student                                            

**4. Remove Non-Analytical Columns**

Remove extraction columns that were useful during scraping but not for modelling: university, filename, pdf_path, status



In [ ]:
#list of known irrelevant columns
drop_cols = [
    "university",   # duplicate / raw name
    "filename",     # extraction artifact
    "pdf_path",     # file reference
    "status"        # extraction status
]

#drop only if they exist
df = df.drop(columns=[col for col in drop_cols if col in df.columns], errors="ignore")

print("Remaining columns:", df.shape[1])

Remaining columns: 48


**4.1 Check Remaining Columns**

In [ ]:
print(df.columns.tolist())

['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'adm_percent_of_first-time_degree/certificate-seeking_students_submitting_sat_scores', 'adm_percent_of_first-time_degree/certificate-seeking_students_submitting_act_scores', 'adm_sat_evidence-based_reading_and_writing_th_percentile_score', 'adm_sat_math_th_percentile_score', 'adm_act_composite_th_percentile_score', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'retention_rate_x', 'student_faculty_ratio', 'aid_any_percent', 'sfa_percent_of_full-time_first-time_undergraduates_awarded_pell_grants', 'aid_avg_a

**5. Column Standardisation & Renaming**

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("'", "")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("%", "pct")
)

In [ ]:
print(df.columns.tolist())

['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'adm_percent_of_first-time_degree/certificate-seeking_students_submitting_sat_scores', 'adm_percent_of_first-time_degree/certificate-seeking_students_submitting_act_scores', 'adm_sat_evidence-based_reading_and_writing_th_percentile_score', 'adm_sat_math_th_percentile_score', 'adm_act_composite_th_percentile_score', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'retention_rate_x', 'student_faculty_ratio', 'aid_any_percent', 'sfa_percent_of_full-time_first-time_undergraduates_awarded_pell_grants', 'aid_avg_a

In [ ]:
#renaming long IPEDS variables
df = df.rename(columns={
    "adm_percent_of_first-time_degree/certificate-seeking_students_submitting_sat_scores": "sat_submission_rate",
    "adm_percent_of_first-time_degree/certificate-seeking_students_submitting_act_scores": "act_submission_rate",
    "adm_sat_evidence-based_reading_and_writing_th_percentile_score":"sat_rw",
    "adm_sat_math_th_percentile_score": "sat_math_75th",
    "adm_act_composite_th_percentile_score": "act_75th",
    "sfa_percent_of_full-time_first-time_undergraduates_awarded_pell_grants":"sfa_pell_grant",
    "graduation_men_bachelors_or_equiv_subcohort_4-yr_institution_completers_of_bachelors_or_equiv_degrees_total_150pct_of_normal_time": "grad_men_completers",
    "graduation_men_bachelors_or_equiv_subcohort_4-yr_institution_transfer-out_students": "grad_men_transfer_out",
    "graduation_total_bachelors_or_equiv_subcohort_4-yr_institution_completers_of_bachelors_or_equiv_degrees_total_150pct_of_normal_time": "grad_total_completers",
    "graduation_total_bachelors_or_equiv_subcohort_4-yr_institution_transfer-out_students": "grad_total_transfer_out",
    "graduation_women_bachelors_or_equiv_subcohort_4-yr_institution_completers_of_bachelors_or_equiv_degrees_total_150pct_of_normal_time": "grad_women_completers",
    "graduation_women_bachelors_or_equiv_subcohort_4-yr_institution_transfer-out_students": "grad_women_transfer_out"
})

In [ ]:
print(df.columns.tolist())

['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'sat_submission_rate', 'act_submission_rate', 'sat_rw', 'sat_math_75th', 'act_75th', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'retention_rate_x', 'student_faculty_ratio', 'aid_any_percent', 'sfa_pell_grant', 'aid_avg_amount', 'grad_men_completers', 'grad_men_transfer_out', 'grad_total_completers', 'grad_total_transfer_out', 'grad_women_completers', 'grad_women_transfer_out', 'retention_rate_y', 'applicants', 'admitted', 'enrolled']


In [ ]:
#save dataset
output_path = "/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_columnrenaming.xlsx"
df.to_excel(output_path, index=False)
print("Saved file:", output_path)

Saved file: /content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_columnrenaming.xlsx


**6. Identify and Remove Duplicate Rows**


In [ ]:
#check duplicate rows
print(df.columns.tolist())


['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'sat_submission_rate', 'act_submission_rate', 'sat_rw', 'sat_math_75th', 'act_75th', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'retention_rate_x', 'student_faculty_ratio', 'aid_any_percent', 'sfa_pell_grant', 'aid_avg_amount', 'grad_men_completers', 'grad_men_transfer_out', 'grad_total_completers', 'grad_total_transfer_out', 'grad_women_completers', 'grad_women_transfer_out', 'retention_rate_y', 'applicants', 'admitted', 'enrolled']


**6.1 Merge the Identified Duplicates**

In [ ]:
#applicants
if "applicants_total" in df.columns and "applicants" in df.columns:
    df["applicants_total"] = df["applicants_total"].fillna(df["applicants"])

#admissions
if "admissions_total" in df.columns and "admitted" in df.columns:
    df["admissions_total"] = df["admissions_total"].fillna(df["admitted"])

#anrolled
if "enrolled_total" in df.columns and "enrolled" in df.columns:
    df["enrolled_total"] = df["enrolled_total"].fillna(df["enrolled"])

#retention rate
if "retention_rate_x" in df.columns and "retention_rate_y" in df.columns:
    df["retention_rate"] = df["retention_rate_x"].fillna(df["retention_rate_y"])
elif "retention_rate_x" in df.columns:
    df["retention_rate"] = df["retention_rate_x"]
elif "retention_rate_y" in df.columns:
    df["retention_rate"] = df["retention_rate_y"]

**6.2 Drop the Redundant Duplicate Columns**

In [ ]:
duplicate_cols_to_drop = [
    "applicants",
    "admitted",
    "enrolled",
    "retention_rate_x",
    "retention_rate_y"
]

df = df.drop(columns=[c for c in duplicate_cols_to_drop if c in df.columns], errors="ignore")

In [ ]:
print("Remaining columns after resolving duplicates:")
print(df.columns.tolist())
print("Dataset shape:", df.shape)

Remaining columns after resolving duplicates:
['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'sat_submission_rate', 'act_submission_rate', 'sat_rw', 'sat_math_75th', 'act_75th', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'student_faculty_ratio', 'aid_any_percent', 'sfa_pell_grant', 'aid_avg_amount', 'grad_men_completers', 'grad_men_transfer_out', 'grad_total_completers', 'grad_total_transfer_out', 'grad_women_completers', 'grad_women_transfer_out', 'retention_rate']
Dataset shape: (300, 44)


In [ ]:
#save the cleaned dataset without duplicate columns
output_path = "/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_no_duplicates.xlsx"
df.to_excel(output_path, index=False)
print("Saved file:", output_path)

Saved file: /content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_no_duplicates.xlsx


**7. Missing Values**

Following data cleaning and validation, missing values within the dataset were addressed to ensure data completeness and improve model reliability.

In [ ]:
#percentage of missing values
missing_percent = (df.isna().sum() / len(df)) * 100

#display missing count and percentage
missing_df = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": missing_percent
}).sort_values(by="Missing %", ascending=False)

#print columns that have a missing percentage greater than 0
print(missing_df[missing_df["Missing %"] > 0])

                         Missing Count  Missing %
grad_men_transfer_out              139  46.333333
grad_total_transfer_out            139  46.333333
grad_women_transfer_out            139  46.333333
sat_rw                              40  13.333333
sat_math_75th                       40  13.333333
act_75th                            39  13.000000
act_submission_rate                 38  12.666667
sat_submission_rate                 38  12.666667
admissions_women                     5   1.666667
enrolled_men                         5   1.666667
enrolled_women                       5   1.666667
admissions_men                       5   1.666667
grad_women_completers                5   1.666667
grad_men_completers                  5   1.666667
retention_rate                       5   1.666667
grad_total_completers                5   1.666667
aid_avg_amount                       5   1.666667
sfa_pell_grant                       5   1.666667
student_faculty_ratio                5   1.666667


**8. Imputation**

Drop High % Missing Values.
These are not relevant to admissions pipeline

In [ ]:
#drop high percentage of missing values
high_missing_cols = [
    "grad_men_transfer_out",
    "grad_total_transfer_out",
    "grad_women_transfer_out"
]

#drop these columns from the DataFrame if they exist
df = df.drop(columns=[c for c in high_missing_cols if c in df.columns], errors="ignore")

In [ ]:
print("Remaining columns after dropping high % missing values:")
print(df.columns.tolist())
print("Dataset shape:", df.shape)

Remaining columns after dropping high % missing values:
['unitid', 'institution_name', 'year', 'level_of_student', 'enrollment_men', 'enrollment_women', 'idx_ef', 'enrollment_asian', 'enrollment_black', 'enrollment_hispanic', 'enrollment_white', 'enrollment_international', 'idx_gr', 'applicants_total', 'applicants_men', 'applicants_women', 'admissions_total', 'admissions_men', 'admissions_women', 'enrolled_total', 'enrolled_men', 'enrolled_women', 'sat_submission_rate', 'act_submission_rate', 'sat_rw', 'sat_math_75th', 'act_75th', 'drvef_full-time_enrollment', 'enrollment_undergrad', 'enrollment_grad', 'institution_control', 'urbanization', 'carnegie_classification', 'student_faculty_ratio', 'aid_any_percent', 'sfa_pell_grant', 'aid_avg_amount', 'grad_men_completers', 'grad_total_completers', 'grad_women_completers', 'retention_rate']
Dataset shape: (300, 41)


**8.1 Impute Numerical Variables (Median)**

In [ ]:
import numpy as np

#select numerical columns
num_cols = df.select_dtypes(include=np.number).columns

#median imputation
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print("Numerical columns imputed using median")

Numerical columns imputed using median


**8.2 Check Remaining Missing Values**

In [ ]:
#check remaining missing values
total_missing = df.isna().sum().sum()

print("Total remaining missing values:", total_missing)

Total remaining missing values: 0


In [ ]:
#save the final dataset
output_path = "/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_cleaned_dataset.xlsx"
df.to_excel(output_path, index=False)
print("Saved file:", output_path)

Saved file: /content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_cleaned_dataset.xlsx


**9. Feature Engineering**

Following data preprocessing, feature engineering was performed to enhance the analytical value of the dataset and better represent the university admissions and enrollment pipeline.

**9.1 Admissions Funnel Features**

In [ ]:
#admission rate (selectivity)
df["admission_rate"] = df["admissions_total"] / df["applicants_total"]

#yield rate (conversion from admitted to enrolled)
df["yield_rate"] = df["enrolled_total"] / df["admissions_total"]

#conversion rate (full pipeline)
df["conversion_rate"] = df["enrolled_total"] / df["applicants_total"]

**9.2 Demographics/Gender-Based Feature**

In [ ]:
#gender balance (proportion of women)
df["gender_balance"] = df["enrolled_women"] / (
    df["enrolled_men"] + df["enrolled_women"]
)

#international student proportion
df["intl_composition"] = df["enrollment_international"] / df["enrollment_undergrad"]

**9.3 Academic & Institutional Features**

In [ ]:
#student-faculty inverse (better interpretability)
df["faculty_student_ratio"] = 1 / df["student_faculty_ratio"]

**9.4 Institutional Composition Features**

In [ ]:

#graduate vs total enrollment
df["grad_composition"] = df["enrollment_grad"] / (
    df["enrollment_undergrad"] + df["enrollment_grad"]
)

**9.5 Financial/Support Indicators**

In [ ]:
#financial aid coverage
df["aid_coverage"] = df["aid_any_percent"] / 100

#sfa_pell-grant
df["pell_share"] = df["sfa_pell_grant"] / 100

**9.6 Handle Infinite Issues**

In [ ]:
import numpy as np

#replace infinite values from division
df = df.replace([np.inf, -np.inf], np.nan)

#fill any new missing values
for col in ["admission_rate", "yield_rate", "conversion_rate"]:
    df[col] = df[col].fillna(df[col].median())

**9.7 Check Enginereed Features**

In [ ]:
print(df[[
    "admission_rate",
    "yield_rate",
    "conversion_rate",
    "gender_balance",
    "intl_composition",
    "faculty_student_ratio",
    "grad_composition",
    "aid_coverage",
    "pell_share"
]].describe())

       admission_rate  yield_rate  conversion_rate  gender_balance  \
count      276.000000  276.000000       276.000000      276.000000   
mean         0.411614    0.394312         0.118769        0.531555   
std          0.303953    0.187263         0.072849        0.049874   
min          0.026946    0.145189         0.013473        0.407055   
25%          0.113770    0.247945         0.051691        0.500303   
50%          0.363380    0.330408         0.108021        0.533148   
75%          0.686865    0.485152         0.166840        0.568344   
max          0.934248    0.970183         0.306752        0.642605   

       intl_composition  faculty_student_ratio  grad_composition  \
count        271.000000             276.000000        276.000000   
mean           0.295152               0.095693          0.363487   
std            0.280477               0.065541          0.185260   
min            0.033318               0.045455          0.112678   
25%            0.100238      

The engineered features directly represent the admissions funnel, enabling the study to model and optimise different stages of the university enrollment process.

In [ ]:
#define file path
output_path = "/content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_feature_engineered.xlsx"

#save the features engineered file
df.to_excel(output_path, index=False)

print("Feature engineered file saved at:", output_path)

Feature engineered file saved at: /content/drive/MyDrive/P2 MDS/03_Data_Preprocessing/final_dataset_feature_engineered.xlsx


In [ ]:
print("Dataset shape:", df.shape)

Dataset shape: (276, 52)
